In [1]:
import csv

def extract_tcal(input_file, output_lcp, output_rcp):
    with open(input_file, 'r') as f, \
         open(output_lcp, 'w', newline='') as lcp_out, \
         open(output_rcp, 'w', newline='') as rcp_out:
        
        lcp_writer = csv.writer(lcp_out)
        rcp_writer = csv.writer(rcp_out)
        # Write headers
        lcp_writer.writerow(['POL', 'FREQ', 'TCAL'])
        rcp_writer.writerow(['POL', 'FREQ', 'TCAL'])
        
        for line in f:
            line = line.strip()
            if not line or line.startswith('*'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            pol, freq, tcal = parts[0], parts[1], parts[2]
            if pol == 'lcp':
                lcp_writer.writerow([pol, freq, tcal])
            elif pol == 'rcp':
                rcp_writer.writerow([pol, freq, tcal])

# Usage calnkc.rxg.work.txt
extract_tcal('calnkc.rxg.work.txt', 'calnkc.rxg.work_lcp_tcal.csv', 'calnkc.rxg.work_rcp_tcal.csv')

# Usage for calnkc.rxg.txt
extract_tcal('calnkc.rxg.txt', 'calnkc.rxg_lcp_tcal.csv', 'calnkc.rxg_rcp_tcal.csv')

In [4]:
#!/usr/bin/env python3
import csv
import re

def parse_xtraclog(logfile):
    onoff_rows = []
    fivept_rows = []

    # Timestamp pattern: YYYY.DDD.HH:MM:SS.ss
    ts_pattern = re.compile(r'^(\d+\.\d+\.\d+:\d+:\d+\.\d+)')

    with open(logfile, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract timestamp
            ts_match = ts_pattern.match(line)
            if not ts_match:
                continue
            timestamp = ts_match.group(1)
            # Remove timestamp from the line
            rest = line[ts_match.end():]

            # ----- ONOFF VAL lines -----
            if rest.startswith('#onoff#VAL'):
                # Remove the tag and split the rest
                data = rest[len('#onoff#VAL'):].strip()
                parts = data.split()
                # Expected parts: source, Az, El, De, I, P, Center, Comp, Tsys, SEFD, Tcal_j, Tcal_r
                if len(parts) >= 12:
                    onoff_rows.append([
                        timestamp,
                        parts[0],  # source
                        parts[1],  # Az
                        parts[2],  # El
                        parts[3],  # De
                        parts[4],  # I
                        parts[5],  # P
                        parts[6],  # Center
                        parts[7],  # Comp
                        parts[8],  # Tsys
                        parts[9],  # SEFD
                        parts[10], # Tcal(j)
                        parts[11]  # Tcal(r)
                    ])
                continue

            # ----- FIVPT offset lines (after peak‑up) -----
            if rest.startswith('#fivpt#offset'):
                data = rest[len('#fivpt#offset'):].strip()
                parts = data.split()
                # Example: ['48.2853', '50.8026', '0.00521', '-0.01527', '1', '1', 'ia', '3c123']
                # The last token is the source, first four are offsets and errors
                if len(parts) >= 5:
                    az_off = parts[0]
                    el_off = parts[1]
                    az_err = parts[2]
                    el_err = parts[3]
                    source = parts[-1]  # last token
                    fivept_rows.append([
                        timestamp, source, az_off, el_off, az_err, el_err
                    ])
                continue

    return onoff_rows, fivept_rows

if __name__ == '__main__':
    log_file = 'xtraclog.log'   # make sure this file is in the current directory
    onoff_data, fivept_data = parse_xtraclog(log_file)

    # Write ONOFF CSV
    with open('onoff_measurements.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'source', 'Az', 'El', 'De', 'I', 'P',
                         'Center', 'Comp', 'Tsys', 'SEFD', 'Tcal(j)', 'Tcal(r)'])
        writer.writerows(onoff_data)

    # Write FIVPT CSV
    with open('fivept_offsets.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'source', 'az_offset_deg', 'el_offset_deg',
                         'az_error_deg', 'el_error_deg'])
        writer.writerows(fivept_data)

    print(f"Extracted {len(onoff_data)} ONOFF measurements and {len(fivept_data)} FIVPT offsets.")

Extracted 1062 ONOFF measurements and 77 FIVPT offsets.
